# 05 - Univariate Filter Feature Selection

This notebook performs the second classification experiment using a
univariate filter feature-selection method.

Each transformed feature is evaluated independently according to its
relationship with the target variable.

The same classifiers, preprocessing strategy, evaluation metrics, and
cross-validation procedure used in the baseline experiment are maintained so
that the effect of feature selection can be evaluated fairly.

In [21]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

pd.set_option("display.max_columns", None)

## 1. Dataset Preparation

The same dataset preparation used in the baseline experiment is applied.

Identifier and constant attributes are removed, while `Attrition` is encoded
as a binary target:

- `No` -> 0
- `Yes` -> 1

Using the same preparation ensures that differences with the baseline
experiment are attributable to feature selection rather than to changes in
the data.

In [22]:
DATA_PATH = "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Preprocessing Definition

The same preprocessing strategy used in the baseline experiment is preserved.

Numerical predictors are standardized and categorical predictors are one-hot
encoded.

The preprocessing transformer will remain inside the machine-learning
pipeline so that it is fitted exclusively using the training data of each
cross-validation fold.

In [23]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical predictors:", len(numerical_features))
print("Categorical predictors:", len(categorical_features))

Numerical predictors: 23
Categorical predictors: 7


In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 3. Univariate Filter Feature Selection

A univariate filter evaluates each feature independently with respect to the
target variable.

In this experiment, `SelectKBest` is combined with the ANOVA F-statistic
(`f_classif`). Each transformed feature receives a score measuring the
difference between the two target classes.

The 15 highest-scoring transformed features are retained.

The selector is included inside the classification pipeline. Therefore,
feature scores and the selected subset are calculated using only the training
portion of each cross-validation fold, preventing information leakage.

In [25]:
N_SELECTED_FEATURES = 15

print(
    f"Number of transformed features to retain: "
    f"{N_SELECTED_FEATURES}"
)

Number of transformed features to retain: 15


## 4. Evaluation Strategy

The same stratified 10-fold cross-validation procedure used in the baseline
experiment is maintained.

For each fold, the complete sequence is:

1. Fit preprocessing using the training fold.
2. Transform the training data.
3. Calculate the univariate feature scores using the training fold.
4. Select the 15 highest-scoring transformed features.
5. Train the classifier using only the selected features.
6. Apply the learned preprocessing and feature selection to the validation
   fold.
7. Evaluate the classifier on the validation fold.

This procedure provides an honest estimation of model performance because the
validation data are not used during preprocessing or feature selection.

In [26]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

## 5. Evaluation Metrics

The same metrics used in the baseline experiment are retained:

- Accuracy
- Precision
- Recall
- F1-score
- Balanced Accuracy

Because the dataset is imbalanced, particular attention is given to recall,
F1-score, and balanced accuracy.

In [27]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy"
}

## 6. Classification Algorithms

The same non-probabilistic classifiers used in the baseline experiment are
evaluated.

Their configurations are kept unchanged so that the only experimental
difference with respect to the baseline is the introduction of univariate
feature selection.

In [28]:
models = {
    "K-NN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "SVM": SVC(),

    "Neural Network": MLPClassifier(
        random_state=42,
        max_iter=2000
    )
}

## 7. Evaluation Function

For each classifier, preprocessing, univariate feature selection, and
classification are combined into a single scikit-learn `Pipeline`.

This guarantees that both preprocessing and feature selection are learned
independently within each training fold.

In [29]:
def evaluate_model(
    name,
    classifier,
    X,
    y,
    cv,
    scoring
):
    """
    Evaluate a classifier using preprocessing,
    univariate feature selection, and stratified
    cross-validation.
    """

    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            (
                "feature_selection",
                SelectKBest(
                    score_func=f_classif,
                    k=N_SELECTED_FEATURES
                )
            ),
            ("classifier", classifier)
        ]
    )

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    result = {
        "Model": name,
        "Selected Features": N_SELECTED_FEATURES
    }

    for metric in scoring.keys():

        metric_values = scores[f"test_{metric}"]

        result[f"{metric}_mean"] = metric_values.mean()
        result[f"{metric}_std"] = metric_values.std()

    return result

## 8. Model Evaluation

Each classifier is evaluated after applying univariate feature selection.

All classifiers use the same number of selected features and the same
cross-validation folds.

In [30]:
results = []

for name, model in models.items():

    print(f"Evaluating {name}...")

    result = evaluate_model(
        name=name,
        classifier=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring
    )

    results.append(result)

print("\nEvaluation completed.")

Evaluating K-NN...
Evaluating Decision Tree...
Evaluating SVM...
Evaluating Neural Network...

Evaluation completed.


## 9. Complete Cross-Validation Results

The following table reports the mean and standard deviation of every
evaluation metric across the ten cross-validation folds.

In [31]:
univariate_results = pd.DataFrame(results)

display(
    univariate_results.round(4)
)

,Model,Selected Features,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,K-NN,15,0.8463,0.0119,0.5761,0.1135,0.2062,0.0743,0.2968,0.0736,0.5877,0.0342
1,Decision Tree,15,0.7789,0.0276,0.3546,0.0509,0.4219,0.0615,0.3809,0.0365,0.6347,0.0246
2,SVM,15,0.8571,0.0125,0.7933,0.1964,0.1692,0.0510,0.2737,0.0708,0.5793,0.0251
3,Neural Network,15,0.8163,0.0161,0.4167,0.0575,0.3293,0.0652,0.3642,0.0527,0.6197,0.0295


## 10. Univariate Selection Performance Summary

The mean performance of each classifier is summarized below.

These values can be directly compared with the baseline experiment because
both experiments use the same classifiers, folds, preprocessing strategy, and
evaluation metrics.

In [32]:
summary_results = univariate_results[
    [
        "Model",
        "Selected Features",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "balanced_accuracy_mean"
    ]
].copy()

summary_results.columns = [
    "Model",
    "Selected Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced Accuracy"
]

summary_results = summary_results.sort_values(
    "F1",
    ascending=False
)

display(
    summary_results.round(4)
)

,Model,Selected Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
1,Decision Tree,15,0.7789,0.3546,0.4219,0.3809,0.6347
3,Neural Network,15,0.8163,0.4167,0.3293,0.3642,0.6197
0,K-NN,15,0.8463,0.5761,0.2062,0.2968,0.5877
2,SVM,15,0.8571,0.7933,0.1692,0.2737,0.5793


## 11. Inspection of the Selected Features

After the honest cross-validation experiment, the preprocessing and
univariate selector are fitted to the complete dataset exclusively for
interpretation.

This additional fit is not used to estimate classification performance.

Its purpose is to inspect the transformed features that receive the highest
univariate scores and understand which employee characteristics appear most
strongly associated with attrition.

In [33]:
inspection_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "feature_selection",
            SelectKBest(
                score_func=f_classif,
                k=N_SELECTED_FEATURES
            )
        )
    ]
)

inspection_pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('feature_selection', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['Age','BusinessTravel','DailyRate',...,'YearsInCurrentRole', 'YearsSinceLastPromotion','YearsWithCurrManager']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be

In [34]:
fitted_preprocessor = (
    inspection_pipeline
    .named_steps["preprocessing"]
)

fitted_selector = (
    inspection_pipeline
    .named_steps["feature_selection"]
)

all_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

selected_mask = fitted_selector.get_support()

selected_feature_names = all_feature_names[
    selected_mask
]

print(
    f"Selected features: "
    f"{len(selected_feature_names)}"
)

display(
    pd.Series(
        selected_feature_names,
        name="Selected transformed feature"
    )
)

Selected features: 15


0                                  num__Age
1                       num__JobInvolvement
2                             num__JobLevel
3                      num__JobSatisfaction
4                        num__MonthlyIncome
5                     num__StockOptionLevel
6                    num__TotalWorkingYears
7                       num__YearsAtCompany
8                   num__YearsInCurrentRole
9                 num__YearsWithCurrManager
10    cat__BusinessTravel_Travel_Frequently
11        cat__JobRole_Sales Representative
12                cat__MaritalStatus_Single
13                         cat__OverTime_No
14                        cat__OverTime_Yes
Name: Selected transformed feature, dtype: str

In [35]:
feature_scores = pd.DataFrame({
    "Feature": all_feature_names,
    "F_score": fitted_selector.scores_,
    "Selected": selected_mask
})

feature_scores = feature_scores.sort_values(
    "F_score",
    ascending=False
)

display(
    feature_scores.head(20).round(4)
)

,Feature,F_score,Selected
49,cat__OverTime_No,94.6565,True
50,cat__OverTime_Yes,94.6565,True
48,cat__MaritalStatus_Single,46.6070,True
16,num__TotalWorkingYears,44.2525,True
7,num__JobLevel,43.2153,True
20,num__YearsInCurrentRole,38.8383,True
9,num__MonthlyIncome,38.4888,True
0,num__Age,38.1759,True
45,cat__JobRole_Sales Representative,37.2128,True
22,num__YearsWithCurrManager,36.7123,True


## 12. Export Results

The cross-validation results, summary table, and univariate feature scores are
saved for later comparison with the baseline and the other feature-selection
experiments.

In [36]:
univariate_results.to_csv(
    "../results/univariate_results.csv",
    index=False
)

summary_results.to_csv(
    "../results/univariate_summary.csv",
    index=False
)

feature_scores.to_csv(
    "../results/univariate_feature_scores.csv",
    index=False
)

print("Univariate feature-selection results saved successfully.")

Univariate feature-selection results saved successfully.


## 13. Comparison with the Baseline

The results obtained after univariate feature selection are compared with the
baseline experiment using all transformed features.

Because both experiments use the same classifiers, preprocessing strategy,
cross-validation folds, and evaluation metrics, differences in performance can
be attributed to the introduction of univariate feature selection.

In [37]:
baseline_summary = pd.read_csv(
    "../results/baseline_summary.csv"
)

baseline_comparison = baseline_summary.copy()
baseline_comparison["Experiment"] = "All Features"

univariate_comparison = summary_results.copy()
univariate_comparison["Experiment"] = "Univariate"

comparison = pd.concat(
    [
        baseline_comparison,
        univariate_comparison
    ],
    ignore_index=True
)

comparison = comparison[
    [
        "Experiment",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy"
    ]
]

display(
    comparison.round(4)
)

,Experiment,Model,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,0.8490,0.5492,0.4219,0.4738,0.6765
1,All Features,Decision Tree,0.7925,0.3604,0.3670,0.3620,0.6206
2,All Features,SVM,0.8653,0.8363,0.2241,0.3461,0.6064
3,All Features,K-NN,0.8442,0.5412,0.1350,0.2115,0.5577
4,Univariate,Decision Tree,0.7789,0.3546,0.4219,0.3809,0.6347
5,Univariate,Neural Network,0.8163,0.4167,0.3293,0.3642,0.6197
6,Univariate,K-NN,0.8463,0.5761,0.2062,0.2968,0.5877
7,Univariate,SVM,0.8571,0.7933,0.1692,0.2737,0.5793


## 14. Main Observations

The univariate filter reduces the transformed representation from 51 features
to 15 selected features.

The effect of this reduction depends strongly on the classifier:

- **K-NN benefits the most from univariate feature selection.** Its F1-score
  increases from 0.2115 to 0.2968, while recall increases from 0.1350 to
  0.2062. Balanced accuracy also improves from 0.5577 to 0.5877.

- **Decision Tree also benefits slightly from feature selection.** Its recall
  increases from 0.3670 to 0.4219 and its F1-score improves from 0.3620 to
  0.3809. Balanced accuracy increases from 0.6206 to 0.6347, although overall
  accuracy decreases slightly.

- **SVM performs worse after univariate selection.** Its F1-score decreases
  from 0.3461 to 0.2737 and balanced accuracy decreases from 0.6064 to 0.5793.

- **Neural Network experiences the largest performance decrease.** Its
  F1-score falls from 0.4738 to 0.3642 and balanced accuracy decreases from
  0.6765 to 0.6197.

- After univariate feature selection, the **Decision Tree becomes the best
  classifier according to F1-score (0.3809) and balanced accuracy (0.6347)**.

- The selected features are consistent with patterns observed during the
  exploratory analysis. In particular, overtime, marital status, frequent
  business travel, job role, age, income, and professional experience appear
  among the highest-scoring predictors.

- The results show that reducing dimensionality does not necessarily improve
  every classifier. Simpler models such as K-NN and Decision Tree benefit from
  the selected subset, whereas SVM and Neural Network lose useful information
  when the representation is reduced to 15 features.